# H&M 고객 데이터 — 전처리(판다스) 4회차 해설/정답


아래 풀이는 **Split -> Fit -> Transform** 순서를 최대한 지키는 버전입니다.  
(전처리에서 기준값을 만드는 순간 = `fit` 이라서, train에서만 만들고 test에는 적용만 해야 누수가 없습니다.)

> 실습 편의상 샘플링(200,000행)을 켜두었습니다. 전체로 돌리려면 샘플 줄을 주석 처리하세요.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("customer_hm.csv")

# 실습 속도 때문에 샘플링 (필요 없으면 주석 처리!)
df = df.sample(200_000, random_state=42).reset_index(drop=True)

print(df.shape)
df.head()


(200000, 6)


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age
0,95a27f54cded740b1f8a71ffc5a10630c788781abae474...,1,1,ACTIVE,Regularly,49
1,b3553fce166fb434ce7862b98cbed186adf77458ee2ec6...,1,1,ACTIVE,Regularly,64
2,ad9e46403030a92dffc30299e1711f9fe791c8ba3a954c...,0,0,ACTIVE,NONE,43
3,9626d0cadb8d86f21fead29f2a8c68d8c175004db8def7...,1,1,ACTIVE,Regularly,35
4,7ec984843e492a4f4fb82e981142d48d7d5741e0e0173c...,1,1,ACTIVE,Regularly,30


### 문제 1 해설 — SPLIT: `Active`를 y로 두고 Train/Test 나누기
- 분류 문제에서 `stratify=y`를 주면 **train/test의 클래스 비율이 비슷하게** 유지됩니다.
- `copy()`는 SettingWithCopy 경고 방지용으로 습관처럼!


In [11]:
y = df["Active"]
X = df.drop(columns=["Active"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 안전하게 copy
X_train = X_train.copy()
X_test = X_test.copy()

print("X_train:", X_train.shape, "X_test:", X_test.shape)


X_train: (160000, 5) X_test: (40000, 5)


### 문제 2 해설 — 결측 플래그 + 최빈값 채우기
- 결측 자체가 신호일 수도 있어서 `..._isna` 플래그를 같이 둡니다.
- 최빈값(mode)은 **train에서만** 계산하고, test에는 그 값을 그대로 적용합니다.


In [ ]:
# 1) 결측 플래그
col = "fashion_news_frequency"
X_train[f"{col}_isna"] = X_train[col].isna().astype(int)
X_test[f"{col}_isna"] = X_test[col].isna().astype(int)

# 2) 결측 채우기 (train에서만 기준값 계산)
# dropna=True 최빈 값을 구할 때 결측치는 계산에서 제외
fn_mode = X_train[col].mode(dropna=True)[0]

X_train[col] = X_train[col].fillna(fn_mode)
X_test[col] = X_test[col].fillna(fn_mode)

X_train[[col, f"{col}_isna"]].head()


,fashion_news_frequency,fashion_news_frequency_isna
119400,Regularly,0
36752,NONE,0
182754,NONE,0
147830,NONE,0
145597,NONE,0


### 문제 3 해설 — 희귀 범주 묶기(Rare)
- 범주가 너무 희귀하면 모델이 그 패턴을 과하게 외우거나(overfit), test에서는 안 나올 수 있어요.
- 그래서 **train에서만 빈도를 보고** 기준을 정한 뒤, 같은 규칙을 test에 적용합니다.


In [4]:
col = "fashion_news_frequency"

fn_counts = X_train[col].value_counts()
rare_fns = fn_counts[fn_counts < 1000].index

# train/test에 동일 적용
X_train[col] = X_train[col].replace(rare_fns, "Rare")
X_test[col] = X_test[col].replace(rare_fns, "Rare")

print("train categories:", X_train[col].value_counts())
print("test categories:", X_test[col].value_counts())


train categories: fashion_news_frequency
NONE         103127
Regularly     56753
Rare            120
Name: count, dtype: int64
test categories: fashion_news_frequency
NONE         25756
Regularly    14215
Rare            29
Name: count, dtype: int64


### 문제 4 해설 — Frequency Encoding
- `value_counts()`로 만든 빈도표는 **train에서만** 만들고(`fit`)
- `map`은 train/test 둘 다 적용(`transform`)합니다.
- test에 train에 없던 범주가 나오면 NaN이 되니 `fillna(0)`으로 안전하게 처리합니다.


In [5]:
# 1) club_member_status 빈도
status_counts = X_train["club_member_status"].value_counts()
X_train["status_freq"] = X_train["club_member_status"].map(status_counts)
X_test["status_freq"] = X_test["club_member_status"].map(status_counts).fillna(0)

# 2) fashion_news_frequency 빈도
fn_counts = X_train["fashion_news_frequency"].value_counts()
X_train["fn_freq"] = X_train["fashion_news_frequency"].map(fn_counts)
X_test["fn_freq"] = X_test["fashion_news_frequency"].map(fn_counts).fillna(0)

X_train[["club_member_status", "status_freq", "fashion_news_frequency", "fn_freq"]].head()


,club_member_status,status_freq,fashion_news_frequency,fn_freq
119400,ACTIVE,149899,Regularly,56753
36752,PRE-CREATE,10033,NONE,103127
182754,ACTIVE,149899,NONE,103127
147830,ACTIVE,149899,NONE,103127
145597,ACTIVE,149899,NONE,103127


### 문제 5 해설 — 파생변수(Feature Engineering)
- `age_band`처럼 **구간화**는 모델이 비선형 패턴을 잡는 데 도움이 될 때가 많아요.
- `news_opt_in`은 `NONE`과 그 외를 0/1로 압축해서 신호를 단순화한 버전.


In [6]:
# 1) age_band
bins = [0, 19, 29, 39, 49, 59, 120]
labels = ["10대", "20대", "30대", "40대", "50대", "60대+"]

X_train["age_band"] = pd.cut(X_train["age"], bins=bins, labels=labels, right=True)
X_test["age_band"] = pd.cut(X_test["age"], bins=bins, labels=labels, right=True)

# 2) is_senior
X_train["is_senior"] = (X_train["age"] >= 60).astype(int)
X_test["is_senior"] = (X_test["age"] >= 60).astype(int)

# 3) news_opt_in
X_train["news_opt_in"] = (X_train["fashion_news_frequency"] != "NONE").astype(int)
X_test["news_opt_in"] = (X_test["fashion_news_frequency"] != "NONE").astype(int)

X_train[["age", "age_band", "is_senior", "fashion_news_frequency", "news_opt_in"]].head()


,age,age_band,is_senior,fashion_news_frequency,news_opt_in
119400,29,20대,0,Regularly,1
36752,33,30대,0,NONE,0
182754,25,20대,0,NONE,0
147830,24,20대,0,NONE,0
145597,39,30대,0,NONE,0


### 문제 6 해설 — 이상치 캡핑 + log1p
- IQR 캡핑은 극단값 영향을 줄이는 가장 무난한 방법입니다.
- `status_freq`, `fn_freq`는 값 스케일이 너무 커서 `log1p`로 눌러주면 학습이 안정적인 경우가 많습니다.
- 핵심: 경계(lo/hi)는 **train에서만 계산**!


In [7]:
# 1) age IQR 캡핑 (경계는 train에서만)
q1, q3 = X_train["age"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

X_train["age_cap"] = X_train["age"].clip(lo, hi)
X_test["age_cap"] = X_test["age"].clip(lo, hi)

# 2) log1p 변환
X_train["age_log1p"] = np.log1p(X_train["age_cap"])
X_test["age_log1p"] = np.log1p(X_test["age_cap"])

X_train["status_freq_log1p"] = np.log1p(X_train["status_freq"])
X_test["status_freq_log1p"] = np.log1p(X_test["status_freq"])

X_train["fn_freq_log1p"] = np.log1p(X_train["fn_freq"])
X_test["fn_freq_log1p"] = np.log1p(X_test["fn_freq"])

X_train[["age", "age_cap", "age_log1p", "status_freq", "status_freq_log1p", "fn_freq", "fn_freq_log1p"]].head()


,age,age_cap,age_log1p,status_freq,status_freq_log1p,fn_freq,fn_freq_log1p
119400,29,29.0,3.401197,149899,11.917724,56753,10.946481
36752,33,33.0,3.526361,10033,9.213735,103127,11.543726
182754,25,25.0,3.258097,149899,11.917724,103127,11.543726
147830,24,24.0,3.218876,149899,11.917724,103127,11.543726
145597,39,39.0,3.688879,149899,11.917724,103127,11.543726


### 문제 7 해설 — 원-핫 인코딩 + 컬럼 정렬
- `get_dummies`를 train/test에 각각 하면 **컬럼 불일치가 거의 무조건 발생**합니다.
- 그래서 test를 train 컬럼에 맞춰 `reindex(..., fill_value=0)` 해주는 게 포인트!


In [ ]:
# 모델 입력에서 빼도 되는 컬럼 정리
drop_cols = ["customer_id", "age", "age_cap", "status_freq", "fn_freq"]
X_train_model = X_train.drop(columns=drop_cols)
X_test_model = X_test.drop(columns=drop_cols)

cat_cols = ["club_member_status", "fashion_news_frequency", "age_band"]

X_train_dum = pd.get_dummies(X_train_model, columns=cat_cols, drop_first=True)
X_test_dum = pd.get_dummies(X_test_model, columns=cat_cols, drop_first=True)

# train 컬럼 기준으로 test 컬럼 맞추기
X_test_dum = X_test_dum.reindex(columns=X_train_dum.columns, fill_value=0)

print("X_train_dum:", X_train_dum.shape)
print("X_test_dum :", X_test_dum.shape)

X_train_dum.head()


X_train_dum: (160000, 16)
X_test_dum : (40000, 16)


,FN,fashion_news_frequency_isna,is_senior,news_opt_in,age_log1p,status_freq_log1p,fn_freq_log1p,club_member_status_LEFT CLUB,club_member_status_PRE-CREATE,fashion_news_frequency_Rare,fashion_news_frequency_Regularly,age_band_20대,age_band_30대,age_band_40대,age_band_50대,age_band_60대+
119400,1,0,0,1,3.401197,11.917724,10.946481,False,False,False,True,True,False,False,False,False
36752,0,0,0,0,3.526361,9.213735,11.543726,False,True,False,False,False,True,False,False,False
182754,0,0,0,0,3.258097,11.917724,11.543726,False,False,False,False,True,False,False,False,False
147830,0,0,0,0,3.218876,11.917724,11.543726,False,False,False,False,True,False,False,False,False
145597,0,0,0,0,3.688879,11.917724,11.543726,False,False,False,False,False,True,False,False,False


### 문제 8 해설 — 스케일링(StandardScaler)
- 스케일러는 평균/표준편차라는 **기준값을 학습(fit)** 합니다.
- 그래서 train에서만 `fit`하고, test에는 `transform`만 합니다.


In [19]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["FN", "age_log1p", "status_freq_log1p", "fn_freq_log1p"]
scale_cols = [c for c in scale_cols if c in X_train_dum.columns]  # 안전장치

scaler = StandardScaler()
X_train_dum[scale_cols] = scaler.fit_transform(X_train_dum[scale_cols])
X_test_dum[scale_cols] = scaler.transform(X_test_dum[scale_cols])

X_train_dum[scale_cols].describe()


,FN,age_log1p,status_freq_log1p,fn_freq_log1p
count,1.600000e+05,1.600000e+05,1.600000e+05,1.600000e+05
mean,-5.684342e-18,1.154632e-17,-9.526824e-17,-2.589040e-17
std,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00
min,-7.412535e-01,-1.903341e+00,-1.115064e+01,-1.937501e+01
25%,-7.412535e-01,-8.790628e-01,2.565749e-01,-1.128313e+00
50%,-7.412535e-01,-1.417029e-01,2.565749e-01,6.434797e-01
75%,1.349066e+00,9.618611e-01,2.565749e-01,6.434797e-01
max,1.349066e+00,2.448140e+00,2.565749e-01,6.434797e-01


### 문제 9 해설 — 로지스틱 회귀로 학습/평가(맛보기)
전처리 결과가 **모델 입력으로 바로 들어가는지** 확인하는 체크용입니다.

- accuracy는 직관적이지만
- 클래스 비율이 한쪽으로 쏠리면 ROC-AUC도 같이 보는 게 좋습니다.


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train_dum, y_train)

pred = model.predict(X_test_dum)
proba = model.predict_proba(X_test_dum)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, proba))


Accuracy: 0.989925
ROC-AUC : 0.9929303307955526
